In [1]:
# Install dependencies
!pip install langchain langchain-community langchain-cohere 
!pip install sentence-transformers faiss-cpu
!pip install langchain-huggingface pypdf langchain-groq
!pip install cohere -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 75.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages

# 🧠 RAG Advanced — 01: Reranking

## The Problem with Basic RAG

When we search the vectorstore, we get top K chunks by **similarity score**.

But similarity ≠ relevance.

```
Query → Vectorstore → [chunk3, chunk1, chunk7, chunk2...] (by similarity)
```

Sometimes chunk #4 actually answers the question better than chunk #1.

## The Solution: Reranking

Add a reranker **after** retrieval to reorder by TRUE relevance.

```
Query
  ↓
Vectorstore → top 10 chunks  (fast but rough)
  ↓
Reranker    → reorders them  (slow but accurate)
  ↓
top 3 chunks → LLM → Answer
```

## Real World Analogy
Google first finds 1000 results (retrieval),
then ranks them by true relevance (reranking). Same idea.

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print("All imports successful!")

/tmp/ipykernel_57/1348894990.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports successful!


## Step 1: Load & Split Document
Same as 01_basics — we need chunks to search through.

In [3]:
# Attention is all you need paper
!wget "https://arxiv.org/pdf/1706.03762" -O attention_is_all_you_need.pdf

# BERT paper 
!wget "https://arxiv.org/pdf/1810.04805" -O bert.pdf

# GPT-3 paper
!wget "https://arxiv.org/pdf/2005.14165" -O gpt3.pdf

# RAG paper itself!
!wget "https://arxiv.org/pdf/2005.11401" -O rag_paper.pdf

--2026-05-23 21:08:02--  https://arxiv.org/pdf/1706.03762
Resolving arxiv.org (arxiv.org)... 151.101.3.42, 151.101.67.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.3.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2215244 (2.1M) [application/pdf]
Saving to: ‘attention_is_all_you_need.pdf’

attention_is_all_yo 100%[===================>]   2.11M  --.-KB/s    in 0.07s   

2026-05-23 21:08:02 (31.9 MB/s) - ‘attention_is_all_you_need.pdf’ saved [2215244/2215244]

--2026-05-23 21:08:02--  https://arxiv.org/pdf/1810.04805
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.3.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 775166 (757K) [application/pdf]
Saving to: ‘bert.pdf’

bert.pdf            100%[===================>] 757.00K  --.-KB/s    in 0.05s   

2026-05-23 21:08:02 (14.2 MB/s) - ‘bert.pdf’ saved [775166/775166]

--2026-05-23 2

In [4]:
for pdf in ["attention_is_all_you_need.pdf", "bert.pdf", "gpt3.pdf", "rag_paper.pdf"]:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    print(f"{pdf}: {len(docs)} pages")

attention_is_all_you_need.pdf: 15 pages
bert.pdf: 16 pages
gpt3.pdf: 75 pages
rag_paper.pdf: 19 pages


In [5]:
# Load document
loader = PyPDFLoader("attention_is_all_you_need.pdf")
docs = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(docs)

print(f"Pages loaded: {len(docs)}")
print(f"Chunks created: {len(chunks)}")

Pages loaded: 15
Chunks created: 52


## Step 2: Create Vectorstore
We embed the chunks and store them in FAISS.
This is our "rough" retriever — fast but not perfectly ranked.

In [6]:
# Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vectorstore
vectorstore = FAISS.from_documents(chunks, embeddings)

# Base retriever — returns top 10 (intentionally high for reranker to work on)
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

print("Vectorstore created!")
print(f"Base retriever will return: 10 chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created!
Base retriever will return: 10 chunks


## Step 3: Add the Reranker

We'll use a local **Cross-Encoder** model from HuggingFace !

### Cross-Encoder vs Bi-Encoder:
| | Bi-Encoder (embeddings) | Cross-Encoder (reranker) |
|--|--|--|
| How | Embeds query & chunk separately | Reads query + chunk TOGETHER |
| Speed | Fast ⚡ | Slower but accurate |
| Use case | First retrieval (top 10) | Reranking (top 10 → top 3) |

The flow:
```
Query → base_retriever (top 10) → CrossEncoder (reorders) → top 3 → LLM
```

`ContextualCompressionRetriever` is the LangChain wrapper that connects
the base retriever + reranker together.

In [7]:
# Local cross-encoder reranker — no API key needed!
encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
)

reranker = CrossEncoderReranker(
    model=encoder,
    top_n=3
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

print("Local reranker ready!")
print("Pipeline: Query → 10 chunks → CrossEncoder → top 3 chunks")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Local reranker ready!
Pipeline: Query → 10 chunks → CrossEncoder → top 3 chunks


## Step 4: Build the RAG Chain with Reranking

Now we connect everything:
- **compression_retriever** → retrieves + reranks
- **ChatGroq** → LLM for answering
- **create_retrieval_chain** → connects them together

In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=512
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the context to answer. If you don't know, say you don't know.\n\nContext:\n{context}"),
    ("human", "{input}"),
])

chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(compression_retriever, chain)

print("RAG chain with reranking ready!")

RAG chain with reranking ready!


## Step 5: Compare Basic RAG vs RAG with Reranking

Let's ask the same question twice:
- Once with the **base retriever** (no reranking)
- Once with the **compression retriever** (with reranking)

This shows us exactly what reranking adds.

In [9]:
question = "what is this project about?"

# Without reranking
basic_chain = create_retrieval_chain(
    base_retriever,
    create_stuff_documents_chain(llm, prompt)
)
basic_result = basic_chain.invoke({"input": question})

# With reranking
reranked_result = rag_chain.invoke({"input": question})

print("=" * 50)
print("WITHOUT Reranking:")
print(basic_result['answer'])
print("=" * 50)
print("WITH Reranking:")
print(reranked_result['answer'])
print("=" * 50)

WITHOUT Reranking:
This project is about the Transformer model, a novel neural network architecture for machine translation and other natural language processing (NLP) tasks. The Transformer model is designed to replace traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs) by relying entirely on an attention mechanism to draw global dependencies between input and output.

The project aims to:

1. Introduce the Transformer model architecture, which consists of an encoder and a decoder, both composed of stacked self-attention and point-wise, fully connected layers.
2. Show that the Transformer model can achieve state-of-the-art results on machine translation tasks, such as English-to-German and English-to-French translation.
3. Demonstrate the effectiveness of the Transformer model in handling long-distance dependencies and parallelization, which makes it more efficient than traditional RNNs and CNNs.
4. Explore the potential of the Transformer model for o

## Why is "Without Reranking" better here?

This is a real and important lesson.

### The numbers tell the story:
- Without reranking → top **10 chunks** go to LLM → more context → detailed answer
- With reranking → top 10 → reranked → only **3 chunks** go to LLM → less context → shorter answer

### So the issue isn't the reranking itself —
it's the **top_n=3** being too aggressive for a "summary" type question.

### Two types of questions behave differently:

| Question Type | Example | Best top_n |
|--------------|---------|------------|
| Summary/Overview | "what is this about?" | 5-7 chunks |
| Specific/Factual | "what is the BLEU score?" | 2-3 chunks |

### Key Lesson:
> Reranking shines on **specific factual questions**, not broad summary questions.
> Let's prove it with a better test question.

In [10]:
question = "what BLEU score did the transformer achieve on English to German translation?"

basic_result = basic_chain.invoke({"input": question})
reranked_result = rag_chain.invoke({"input": question})

print("=" * 50)
print("WITHOUT Reranking:")
print(basic_result['answer'])
print("=" * 50)
print("WITH Reranking:")
print(reranked_result['answer'])
print("=" * 50)

WITHOUT Reranking:
The Transformer achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task.
WITH Reranking:
The Transformer achieved a BLEU score of 28.4 on the English-to-German translation task.


## Analysis: Specific Question Results

Both got the correct answer: **28.4 BLEU score** ✅

But notice:
- Without reranking → "on the **WMT 2014** English-to-German translation task"
- With reranking → "on the English-to-German translation task"


## Final Summary: When to Use Reranking

### Why both answers are the same here:
The paper is small (~15 pages) → base retriever already finds the right chunks easily.

### Reranking truly shines when:
| Scenario | Without Reranking | With Reranking |
|----------|------------------|----------------|
| Small doc (< 20 pages) | ✅ Works fine | Overkill |
| Large doc (100+ pages) | ❌ Misses answers | ✅ Finds buried info |
| Specific factual questions | ⚠️ Hit or miss | ✅ Precise |
| Summary questions | ✅ Good | ⚠️ Too compressed |
| Multiple documents | ❌ Gets confused | ✅ Ranks best source |

### Rule of thumb:
> If your vectorstore has **more than 50 chunks** and questions are **specific** → add reranking.
> Always set `base_retriever k = 3x` your `reranker top_n`.

### Our pipeline is now:
```
PDF → Chunks → Embeddings → FAISS
         ↓
Query → base_retriever (k=10) → CrossEncoder → top 3 → LLM → Answer
```

## Upgrade: Multi-Document RAG

Real RAG systems don't search ONE document.
They search across MULTIPLE documents and find the best answer.

### Our setup:
- attention_is_all_you_need.pdf (15 pages)
- bert.pdf (16 pages)  
- gpt3.pdf (75 pages)
- rag_paper.pdf (19 pages)

Total: ~125 pages — now reranking will actually matter!

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdfs = [
    "attention_is_all_you_need.pdf",
    "bert.pdf",
    "gpt3.pdf",
    "rag_paper.pdf"
]

all_docs = []
for pdf in pdfs:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    # Add source metadata to each doc
    for doc in docs:
        doc.metadata["source"] = pdf
    all_docs.extend(docs)
    print(f"✅ {pdf}: {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(all_docs)

print(f"\nTotal pages: {len(all_docs)}")
print(f"Total chunks: {len(chunks)}")

✅ attention_is_all_you_need.pdf: 15 pages
✅ bert.pdf: 16 pages
✅ gpt3.pdf: 75 pages
✅ rag_paper.pdf: 19 pages

Total pages: 125
Total chunks: 544


## Rebuild Vectorstore with All Documents

- Now we embed all 544 chunks into FAISS.
- The base retriever will search across ALL 4 papers.

In [12]:
# Rebuild vectorstore with all docs
vectorstore = FAISS.from_documents(chunks, embeddings)

# Base retriever — k=10 intentionally high for reranker
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

# Reranker stays the same
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

# Rebuild chains
basic_chain = create_retrieval_chain(
    base_retriever,
    create_stuff_documents_chain(llm, prompt)
)

rag_chain = create_retrieval_chain(
    compression_retriever,
    create_stuff_documents_chain(llm, prompt)
)

print(f"Vectorstore rebuilt with {len(chunks)} chunks!")
print("Ready to search across 4 papers!")

Vectorstore rebuilt with 544 chunks!
Ready to search across 4 papers!


## Test: Multi-Document Reranking

Let's ask a question that requires finding the right answer
across 4 different papers.

The base retriever might return chunks from wrong papers.
The reranker should surface the most relevant one.

In [13]:
question = "what is the difference between GPT and BERT in terms of training approach?"

basic_result = basic_chain.invoke({"input": question})
reranked_result = rag_chain.invoke({"input": question})

print("=" * 50)
print("WITHOUT Reranking:")
print(basic_result['answer'])
print("\nSources used:")
for doc in basic_result['context']:
    print(f"  - {doc.metadata['source']} | page {doc.metadata.get('page', '?')}")

print("=" * 50)
print("WITH Reranking:")
print(reranked_result['answer'])
print("\nSources used:")
for doc in reranked_result['context']:
    print(f"  - {doc.metadata['source']} | page {doc.metadata.get('page', '?')}")
print("=" * 50)
for i, doc in enumerate(reranked_result['context']):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source']} | Page: {doc.metadata.get('page', '?')}")
    print(f"Content preview: {doc.page_content[:150]}")
    print(f"Content length: {len(doc.page_content)} chars")
print("=" * 50)

WITHOUT Reranking:
According to the text, the main differences between GPT and BERT in terms of training approach are:

1. **Training data**: GPT was trained on the BooksCorpus (800M words), while BERT was trained on the BooksCorpus (800M words) and Wikipedia (2,500M words).
2. **Training direction**: GPT uses a left-to-right Transformer LM, while BERT uses a bidirectional Transformer.
3. **Sentence separator and classifier token**: GPT uses a sentence separator ([SEP]) and classifier token ([CLS]) which are only introduced at fine-tuning time, while BERT learns [SEP], [CLS], and sentence A/B embeddings during pre-training.
4. **Training steps and batch size**: GPT was trained for 1M steps with a batch size of 32,000 words, while BERT was trained for an unspecified number of steps with a batch size of 128,000 words.
5. **Learning rate**: GPT uses a fixed learning rate of 5e-5 for all fine-tuning experiments, while BERT chooses a task-specific fine-tuning learning rate which performs th

## Analysis: Reranking Finally Shows Its Power!

### Without Reranking — 10 chunks from 3 different sources:
- bert.pdf | page 13 (×4)
- bert.pdf | page 12, 5, 7, 2
- gpt3.pdf | page 17, 19

→ Noisy, redundant chunks, pulled from wrong sources too

### With Reranking — 3 chunks, laser focused:
- bert.pdf | page 13 (×3)

→ Reranker identified that page 13 of BERT paper is WHERE the comparison lives

### The answers:
- Without reranking → 5 points but some are from wrong context
- With reranking → 5 points, all precise, all from the right source

### Key Lesson:
> Reranking doesn't just improve quality — it removes noise.
> With 544 chunks across 4 papers, the reranker found exactly
> the right 3 chunks from the right page.

### This is why production RAG systems always use reranking:
- Less hallucination ✅
- More precise sources ✅
- Less tokens sent to LLM = cheaper ✅